# This model is trained on (2020/01/01  to  2023/12/30) 48 month data with STATIC MASK

## Masked on geboc (detailed) depth

In [6]:
# pip install pytorch-lightning

In [2]:
import torch

# Check if CUDA is available
cuda_available = torch.cuda.is_available()
print(f"Is CUDA available? {cuda_available}")

# If it is available, check how many GPUs are detected
if cuda_available:
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    print(f"Current GPU Name: {torch.cuda.get_device_name(0)}")

# Check the CUDA version PyTorch was built with

print(f"PyTorch CUDA Version: {torch.version.cuda}")

Is CUDA available? True
Number of GPUs: 2
Current GPU Name: NVIDIA GeForce GTX 1080 Ti
PyTorch CUDA Version: 12.4


In [3]:
# =============================================================================
# MASTER CONFIGURATION CELL
# =============================================================================
import torch
import os

# --- Core Parameters ---
# Change this value to 24, 36, 72, etc., to test different models
LOOKBACK_HOURS = 96
# Forecast period is now 7 days (7 * 24 = 168)
FORECAST_HORIZON_HOURS = 168

# --- File Paths ---
# MODIFICATION: Updated to the new file paths you provided
RAW_DATA_PATH = r'c:\Users\user\Documents\GitHub\Wave-Prediction\dAtA\cmems_mod_ibi_wav_my_0.027deg_PT1H-i_multi-vars_11.00W-8.53W_38.50N-40.47N_2020-01-01-2023-12-30.nc'
STATIC_DATA_PATH = r'C:\Users\user\Documents\GitHub\Wave-Prediction\dAtA\cmems_GEBCO_resampled.nc'

# MODIFICATION: File paths are now dynamic to keep experiments separate
BASE_DIR = r'D:\babe_prediction'
PROCESSED_DATA_DIR = os.path.join(BASE_DIR, f'processed_data_lookback_{LOOKBACK_HOURS}_static')
# This path is now used by the ModelCheckpoint callback in Lightning
MODEL_SAVE_DIR = 'models/'
MODEL_SAVE_PATH = os.path.join(MODEL_SAVE_DIR, f'convlstm_lookback_{LOOKBACK_HOURS}_forecast_{FORECAST_HORIZON_HOURS}_static.pth')


# --- Feature Engineering ---
# These are the TIME-VARYING features
VARS_TO_USE = ['VCMX','VSDmag', 'VTM10', 'VTM02', 'VTM01_WW', 'VTM01_SW1', 'VMXL', 'VHM0_WW', 'VHM0_SW1']
TARGET_VAR = 'VCMX'
# MODIFICATION: Total input channels = 9 time-varying + 2 static (depth, mask)
INPUT_CHANNELS = len(VARS_TO_USE) + 2

# --- Training Hyperparameters ---
LEARNING_RATE = 1e-5
BATCH_SIZE = 16
EPOCHS = 50
# MODIFICATION: Early stopping patience is now a configurable parameter
EARLY_STOPPING_PATIENCE = 5

# --- System Configuration ---
NUM_WORKERS = 0  # Set to 0 if you encounter multiprocessing issues, 4+ for speed
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Print a summary of the current configuration ---
print("--- Configuration Summary ---")
print(f"Lookback Period: {LOOKBACK_HOURS} hours")
print(f"Forecast Horizon: {FORECAST_HORIZON_HOURS} hours")
print(f"Input Channels: {INPUT_CHANNELS} ({len(VARS_TO_USE)} time-varying + 2 static)")
print(f"Processed Data Path: {PROCESSED_DATA_DIR}")
print(f"Model Save Directory: {MODEL_SAVE_DIR}")
print(f"Using Device: {device}")
print("---------------------------")

--- Configuration Summary ---
Lookback Period: 96 hours
Forecast Horizon: 168 hours
Input Channels: 11 (9 time-varying + 2 static)
Processed Data Path: D:\babe_prediction\processed_data_lookback_96_static
Model Save Directory: models/
Using Device: cuda
---------------------------


# STEP 1: PRE-PROCESSING SCRIPT (with Static Features)

In [16]:
# =============================================================================
# =============================================================================
import os
import pickle
import xarray as xr
import numpy as np
import torch
from sklearn.preprocessing import MinMaxScaler
from tqdm.auto import tqdm
import warnings
import shutil

warnings.filterwarnings('ignore')

if os.path.exists(PROCESSED_DATA_DIR):
    print(f"Removing old processed data directory: {PROCESSED_DATA_DIR}")
    shutil.rmtree(PROCESSED_DATA_DIR)

print("\n--- Starting Full Pre-processing Workflow ---")

# Step 1: Load and Clean Data
print("\n[Step 1/5] Loading and cleaning all data...")
ds_raw = xr.open_dataset(RAW_DATA_PATH)
ds_raw['VSDmag'] = np.sqrt(ds_raw['VSDX']**2 + ds_raw['VSDY']**2)
ds_clean = ds_raw[VARS_TO_USE].astype(np.float32).fillna(0)
print("✅ Time-varying data loaded and cleaned.")

ds_static = xr.open_dataset(STATIC_DATA_PATH)
ocean_depth = ds_static['deptho'].values.astype(np.float32)
ocean_mask = ds_static['mask'].values.astype(np.float32)
print("✅ Static data loaded and cleaned from new GEBCO file.")

# Step 2: Define Data Splits
print("\n[Step 2/5] Splitting data into train, validation, and test sets...")
ds_train = ds_clean.sel(time=slice('2020-01-01', '2022-12-31'))
ds_val = ds_clean.sel(time=slice('2023-01-01', '2023-06-30'))
ds_test = ds_clean.sel(time=slice('2023-07-01', '2023-12-30'))
ds_splits = {'train': ds_train, 'val': ds_val, 'test': ds_test}
print(f"Train split: {len(ds_train.time)} time steps")
print(f"Validation split: {len(ds_val.time)} time steps")
print(f"Test split: {len(ds_test.time)} time steps")

# Step 3: Create and Save Scalers
print("\n[Step 3/5] Fitting scalers on TRAINING data only...")
scalers = {}
for var in tqdm(VARS_TO_USE, desc="Fitting Time-Varying Scalers"):
    data_to_fit = ds_train[var].values.reshape(-1, 1)
    scaler = MinMaxScaler()
    scaler.fit(data_to_fit)
    scalers[var] = scaler

print("Fitting static feature scalers...")
depth_scaler = MinMaxScaler()
depth_scaler.fit(ocean_depth.reshape(-1, 1))
scalers['ocean_depth'] = depth_scaler

os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)
scaler_path = os.path.join(PROCESSED_DATA_DIR, 'scalers_mask.pkl')
with open(scaler_path, 'wb') as f:
    pickle.dump(scalers, f)
print(f"✅ All scalers fitted and saved to '{scaler_path}'")

# Step 4: Scale Static Features
print("\n[Step 4/5] Scaling static features...")
scaled_depth = scalers['ocean_depth'].transform(ocean_depth.reshape(-1, 1)).reshape(ocean_depth.shape)
scaled_depth_ch = np.expand_dims(scaled_depth, axis=0)
ocean_mask_ch = np.expand_dims(ocean_mask, axis=0)
static_features_np = np.concatenate([scaled_depth_ch, ocean_mask_ch], axis=0)
print("✅ Static features scaled.")

# Step 5: Process and Save Samples
print("\n[Step 5/5] Generating and saving individual samples...")
total_window_size = LOOKBACK_HOURS + FORECAST_HORIZON_HOURS

for split_name, ds_split in ds_splits.items():
    print(f"\n--- Processing '{split_name}' split ---")
    split_dir = os.path.join(PROCESSED_DATA_DIR, split_name)
    os.makedirs(split_dir, exist_ok=True)
    
    num_sequences = len(ds_split['time']) - total_window_size
    for i in tqdm(range(num_sequences), desc=f"Saving {split_name} samples"):
        window_slice = ds_split.isel(time=slice(i, i + total_window_size))
        
        scaled_window_vars = []
        for var in VARS_TO_USE:
            data = window_slice[var].values
            scaled_data = scalers[var].transform(data.reshape(-1, 1)).reshape(data.shape)
            scaled_window_vars.append(scaled_data)
        
        scaled_window_tensor = np.stack(scaled_window_vars, axis=1)

        X_tv_np = scaled_window_tensor[:LOOKBACK_HOURS, :, :, :]
        y_np = scaled_window_tensor[LOOKBACK_HOURS:, VARS_TO_USE.index(TARGET_VAR), :, :]

        static_features_tiled = np.tile(static_features_np, (LOOKBACK_HOURS, 1, 1, 1))
        X_final_np = np.concatenate([X_tv_np, static_features_tiled], axis=1)

        X = torch.from_numpy(X_final_np.astype(np.float32))
        y = torch.from_numpy(y_np.astype(np.float32))
        
        sample_path = os.path.join(split_dir, f'sample_{i:06d}.pt')
        torch.save((X, y), sample_path)

print(f"\n\n✅ Pre-processing complete! Data is ready at: '{PROCESSED_DATA_DIR}'")

Removing old processed data directory: D:\babe_prediction\processed_data_lookback_96_static

--- Starting Full Pre-processing Workflow ---

[Step 1/5] Loading and cleaning all data...
✅ Time-varying data loaded and cleaned.
✅ Static data loaded and cleaned from new GEBCO file.

[Step 2/5] Splitting data into train, validation, and test sets...
Train split: 26304 time steps
Validation split: 4344 time steps
Test split: 4392 time steps

[Step 3/5] Fitting scalers on TRAINING data only...


Fitting Time-Varying Scalers:   0%|          | 0/9 [00:00<?, ?it/s]

Fitting static feature scalers...
✅ All scalers fitted and saved to 'D:\babe_prediction\processed_data_lookback_96_static\scalers_mask.pkl'

[Step 4/5] Scaling static features...
✅ Static features scaled.

[Step 5/5] Generating and saving individual samples...

--- Processing 'train' split ---


Saving train samples:   0%|          | 0/26040 [00:00<?, ?it/s]


--- Processing 'val' split ---


Saving val samples:   0%|          | 0/4080 [00:00<?, ?it/s]


--- Processing 'test' split ---


Saving test samples:   0%|          | 0/4128 [00:00<?, ?it/s]



✅ Pre-processing complete! Data is ready at: 'D:\babe_prediction\processed_data_lookback_96_static'


# STEP 2: TRAINING SCRIPT

In [4]:
# ==============================================================================
# ==============================================================================
import os
import pickle
import glob
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import time
import numpy as np

# --- Dataset and Model Classes ---
class PreprocessedWaveDataset(Dataset):
    def __init__(self, split_dir):
        self.file_paths = sorted(glob.glob(os.path.join(split_dir, '*.pt')))
    def __len__(self):
        return len(self.file_paths)
    def __getitem__(self, idx):
        return torch.load(self.file_paths[idx])

class ConvLSTMCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size, bias):
        super(ConvLSTMCell, self).__init__(); self.input_dim, self.hidden_dim, self.kernel_size, self.bias = input_dim, hidden_dim, kernel_size, bias; self.padding = kernel_size[0] // 2; self.conv = nn.Conv2d(self.input_dim + self.hidden_dim, 4 * self.hidden_dim, self.kernel_size, padding=self.padding, bias=self.bias)
    def forward(self, x, h_c): h, c = h_c; combined = torch.cat([x, h], dim=1); cc = self.conv(combined); i, f, o, g = torch.split(cc, self.hidden_dim, dim=1); i, f, o, g = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o), torch.tanh(g); c_n = f * c + i * g; h_n = o * torch.tanh(c_n); return h_n, c_n
    def init_hidden(self, b, i, d): h, w = i; return (torch.zeros(b, self.hidden_dim, h, w, device=d), torch.zeros(b, self.hidden_dim, h, w, device=d))

class ConvLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size, num_layers, batch_first=True, bias=True):
        super(ConvLSTM, self).__init__(); self.batch_first, self.num_layers = batch_first, num_layers; hidden_dims = [hidden_dim] * num_layers if isinstance(hidden_dim, int) else hidden_dim; cell_list = [];
        for i in range(self.num_layers): cur_input_dim = input_dim if i == 0 else hidden_dims[i - 1]; cell_list.append(ConvLSTMCell(cur_input_dim, hidden_dims[i], kernel_size, bias)); self.cell_list = nn.ModuleList(cell_list)
    def forward(self, x, h_c=None):
        b, s_l, _, h, w = x.size();
        if h_c is None: h_c = self._init_hidden(b, (h, w), x.device)
        cur_in = x
        for l_idx in range(self.num_layers):
            h, c = h_c[l_idx]; output_inner = []
            for t in range(s_l): h, c = self.cell_list[l_idx](cur_in[:, t, :, :, :], [h, c]); output_inner.append(h)
            cur_in = torch.stack(output_inner, dim=1)
        return cur_in, [h, c]
    def _init_hidden(self, b, i, d): return [cell.init_hidden(b, i, d) for cell in self.cell_list]

class ConvLSTMNet(nn.Module):
    def __init__(self, input_dim, forecast_horizon, hidden_dims=[64, 32], kernel_size=(3, 3)):
        super(ConvLSTMNet, self).__init__(); self.cl1 = ConvLSTM(input_dim, hidden_dims[0], kernel_size, 1, batch_first=True); self.cl2 = ConvLSTM(hidden_dims[0], hidden_dims[1], kernel_size, 1, batch_first=True); self.output_conv = nn.Conv2d(hidden_dims[1], forecast_horizon, kernel_size=(1, 1), padding='same')
    def forward(self, x_seq): l1_o, _ = self.cl1(x_seq); l2_o, _ = self.cl2(l1_o); return self.output_conv(l2_o[:, -1, :, :, :])

def train_model(model, train_loader, val_loader, device, epochs, lr, patience, model_path):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr); loss_fn = nn.MSELoss(); best_val_loss = float('inf'); scaler = torch.cuda.amp.GradScaler(); epochs_no_improve = 0
    print("\n--- Starting Model Training ---")
    for epoch in range(epochs):
        start_time = time.time(); model.train(); total_train_loss = 0.0; train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Training]")
        for X, y in train_pbar:
            X, y = X.to(device), y.to(device); optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(): predicted = model(X); target = y; loss = loss_fn(predicted, target)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update(); total_train_loss += loss.item(); train_pbar.set_postfix({'loss': f'{loss.item():.6f}'})
        avg_train_loss = total_train_loss / len(train_loader); model.eval(); total_val_loss = 0.0
        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(device), y.to(device)
                with torch.cuda.amp.autocast(): predicted = model(X); target = y; loss = loss_fn(predicted, target)
                total_val_loss += loss.item()
        avg_val_loss = total_val_loss / len(val_loader); epoch_time = time.time() - start_time
        print(f"Epoch {epoch+1}/{epochs} - {epoch_time:.1f}s - Train Loss: {avg_train_loss:.6f} - Val Loss: {avg_val_loss:.6f}")
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss; epochs_no_improve = 0; torch.save(model.state_dict(), model_path); print(f"✅ New best model saved with validation loss: {best_val_loss:.6f}")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience: print(f"Early stopping triggered after {epoch+1} epochs."); break
    print(f"\n✅ Training completed. Best model saved to {model_path}"); model.load_state_dict(torch.load(model_path)); return model

In [8]:
# --- Main Execution Logic ---
if __name__ == '__main__':
    train_dir = os.path.join(PROCESSED_DATA_DIR, 'train')
    val_dir = os.path.join(PROCESSED_DATA_DIR, 'val')
    train_dataset = PreprocessedWaveDataset(train_dir)
    val_dataset = PreprocessedWaveDataset(val_dir)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True if NUM_WORKERS > 0 else False)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True if NUM_WORKERS > 0 else False)
    print(f"\nDataLoaders created with {len(train_dataset)} training samples and {len(val_dataset)} validation samples.")
    
    model = ConvLSTMNet(input_dim=INPUT_CHANNELS, forecast_horizon=FORECAST_HORIZON_HOURS)
    model.to(device)
    print(f"Model built with {INPUT_CHANNELS} input channels and a {FORECAST_HORIZON_HOURS}-hour forecast horizon.")

    trained_model = train_model(
        model=model, train_loader=train_loader, val_loader=val_loader, device=device,
        epochs=EPOCHS, lr=LEARNING_RATE, patience=EARLY_STOPPING_PATIENCE, model_path=MODEL_SAVE_PATH
    )


DataLoaders created with 26040 training samples and 4080 validation samples.
Model built with 11 input channels and a 168-hour forecast horizon.

--- Starting Model Training ---


C:\Users\user\AppData\Local\Temp\ipykernel_5088\3821595659.py:50: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  optimizer = torch.optim.Adam(model.parameters(), lr=lr); loss_fn = nn.MSELoss(); best_val_loss = float('inf'); scaler = torch.cuda.amp.GradScaler(); epochs_no_improve = 0


Epoch 1/50 [Training]:   0%|          | 0/1628 [00:00<?, ?it/s]

C:\Users\user\AppData\Local\Temp\ipykernel_5088\3821595659.py:21: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(self.file_paths[idx])
C:\Users\user\AppData

OutOfMemoryError: CUDA out of memory. Tried to allocate 14.00 MiB. GPU 0 has a total capacity of 11.00 GiB of which 0 bytes is free. Of the allocated memory 8.38 GiB is allocated by PyTorch, and 1.95 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [4]:
# =============================================================================
# STEP 2: EXECUTE THE TRAINING SCRIPT
# =============================================================================

# This command runs the train.py script using the system's Python interpreter.
# The script will handle the multi-GPU training.
# Make sure your conda/virtual environment is selected as the kernel for this notebook.
import sys

# Get the absolute path to the Python executable that this notebook is using
python_executable = sys.executable
print(f"--- Using this specific Python interpreter to run the script: ---")
print(python_executable)
print("-" * 60)

# Execute the script using the correct interpreter path
!"{python_executable}" train.py
print("\n--- Training script finished ---")

# Manually find the best model path after training to use in the next step
# The checkpoint callback in the script prints this path upon completion.
# We recreate it here based on the config.
# Note: A more robust way would be to parse the output or read from a log file.
try:
    # Find the saved checkpoint file
    checkpoint_dir = os.path.dirname(MODEL_SAVE_PATH)
    list_of_files = glob.glob(os.path.join(checkpoint_dir, '*.ckpt'))
    if not list_of_files:
        raise FileNotFoundError
    best_model_path = max(list_of_files, key=os.path.getctime)
    print(f"Found best model at: {best_model_path}")
except (FileNotFoundError, NameError):
    print("Warning: Could not automatically find the best model path. You may need to set it manually for the evaluation step.")
    best_model_path = "" # Set manually if needed

--- Using this specific Python interpreter to run the script: ---
C:\Users\user\.conda\envs\gpu-env\python.exe
------------------------------------------------------------

--- Training script finished ---

--- Starting DDP training on GPUs 0 and 1 ---

--- Starting DDP training on GPUs 0 and 1 ---


Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/2
[W1006 14:56:20.000000000 socket.cpp:518] [c10d] The server socket has failed to bind to [DESKTOP-192NM7K]:12355 (system error: 10048 - Only one usage of each socket address (protocol/network address/port) is normally permitted.).
[W1006 14:56:20.000000000 socket.cpp:518] [c10d] The server socket has failed to bind to DESKTOP-192NM7K:12355 (system error: 10013 - An attempt was made to access a socket in a way forbidden by its access permissions.).
[E1006 14:56:20.000000000 socket.cpp:554] [c10d] The server socket has failed to listen on any local network address.
Traceback (most recent call last):
  File "C:\Users\user\Documents\GitHub\Wave-Prediction\train.py", line 150, in <module>
    main()
  File "C:\Users\user\Documents\GitHub\Wave-Prediction\train.py", line 146, in main
    trainer.fit(lightning_m

In [2]:
!python test1.py

--> Starting process on Rank 1.
--> Starting process on Rank 0.
--- Starting minimal DDP test with TCP init ---

FAILED: The test script crashed. See error below.


W0930 10:32:21.488000 21564 site-packages\torch\multiprocessing\spawn.py:160] Terminating process 3880 via signal SIGTERM
Traceback (most recent call last):
  File "C:\Users\user\Documents\GitHub\Wave-Prediction\test1.py", line 36, in main
    mp.spawn(run,
  File "C:\Users\user\.conda\envs\gpu-env\lib\site-packages\torch\multiprocessing\spawn.py", line 328, in spawn
    return start_processes(fn, args, nprocs, join, daemon, start_method="spawn")
  File "C:\Users\user\.conda\envs\gpu-env\lib\site-packages\torch\multiprocessing\spawn.py", line 284, in start_processes
    while not context.join():
  File "C:\Users\user\.conda\envs\gpu-env\lib\site-packages\torch\multiprocessing\spawn.py", line 203, in join
    raise ProcessRaisedException(msg, error_index, failed_process.pid)
torch.multiprocessing.spawn.ProcessRaisedException: 

-- Process 1 terminated with the following error:
Traceback (most recent call last):
  File "C:\Users\user\.conda\envs\gpu-env\lib\site-packages\torch\multiproce

# STEP 3: EVALUATION AND VISUALIZATION SCRIPT

In [ ]:
# =============================================================================
# STEP 3: EVALUATION AND VISUALIZATION SCRIPT
# =============================================================================
import warnings
warnings.filterwarnings('ignore')

import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import os
import pickle
from torch.utils.data import DataLoader

print("--- Starting Final Evaluation on the Test Set ---\n")

# --- Load Scalers ---
try:
    scaler_path = os.path.join(PROCESSED_DATA_DIR, 'scalers_mask.pkl')
    with open(scaler_path, 'rb') as f:
        scalers = pickle.load(f)
    print("✅ Scalers loaded successfully.")
except FileNotFoundError:
    raise SystemExit(f"❌ ERROR: Scalers file not found at {scaler_path}.")

# --- Prepare Test Loader ---
test_dir = os.path.join(PROCESSED_DATA_DIR, 'test')
test_dataset = PreprocessedWaveDataset(test_dir)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
print("✅ Test data loader prepared.")

# --- Load the Best Model from Checkpoint ---
# Use the path saved by the checkpoint callback during training
try:
    print(f"Attempting to load best model from: {best_model_path}")
    model = WavePredLightningModule.load_from_checkpoint(
        checkpoint_path=best_model_path,
        input_dim=INPUT_CHANNELS,
        forecast_horizon=FORECAST_HORIZON_HOURS,
        lr=LEARNING_RATE
    )
    model.to(device)
    model.eval()
    print("✅ Best model loaded successfully from checkpoint.")
except NameError:
     raise SystemExit(f"❌ ERROR: 'best_model_path' not found. Please ensure the training cell ran successfully.")
except FileNotFoundError:
    raise SystemExit(f"❌ ERROR: Model checkpoint file not found at '{best_model_path}'.")


# --- Evaluation and Visualization Functions ---
def evaluate_and_visualize(model, test_loader, device, scalers, target_var):
    model.eval()
    all_predictions = []
    all_targets = []
    
    print("\n--- Generating predictions for the test set... ---\n")
    with torch.no_grad():
        for X, y in tqdm(test_loader, desc="Evaluating Test Set"):
            X, y = X.to(device), y.to(device)
            predictions_scaled = model(X)
            
            target_scaler = scalers[target_var]
            
            # Reshape for inverse_transform
            pred_flat = predictions_scaled.cpu().numpy().reshape(-1, 1)
            target_flat = y.cpu().numpy().reshape(-1, 1)
            
            # Inverse transform
            pred_original = target_scaler.inverse_transform(pred_flat).reshape(predictions_scaled.shape)
            target_original = target_scaler.inverse_transform(target_flat).reshape(y.shape)
            
            all_predictions.append(pred_original)
            all_targets.append(target_original)

    all_predictions = np.concatenate(all_predictions, axis=0)
    all_targets = np.concatenate(all_targets, axis=0)

    rmse = np.sqrt(np.mean((all_predictions - all_targets) ** 2))
    mae = np.mean(np.abs(all_predictions - all_targets))
    
    print(f"\n📊 Test Results (all {FORECAST_HORIZON_HOURS} hours):")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  MAE:  {mae:.4f}")
    
    # Select first 2 samples for visualization
    vis_data = {'predictions': all_predictions[:2], 'targets': all_targets[:2]}
    return vis_data, all_predictions.flatten(), all_targets.flatten()

def plot_visual_comparison(vis_data, save_path, timesteps_to_plot):
    num_timesteps = len(timesteps_to_plot)
    for i in range(len(vis_data['predictions'])):
        fig, axes = plt.subplots(num_timesteps, 3, figsize=(18, 5 * num_timesteps), squeeze=False)
        plt.suptitle(f"Visual Comparison for Sample {i+1}", fontsize=18, y=0.99)
        for row, t_idx in enumerate(timesteps_to_plot):
            pred_s, targ_s = vis_data['predictions'][i][t_idx, :, :], vis_data['targets'][i][t_idx, :, :]
            v_max = max(np.max(pred_s), np.max(targ_s), 0.1)
            im1 = axes[row, 0].imshow(targ_s, cmap='viridis', vmin=0, vmax=v_max)
            axes[row, 0].set_title(f'Target (Hour {t_idx+1})'); fig.colorbar(im1, ax=axes[row, 0])
            im2 = axes[row, 1].imshow(pred_s, cmap='viridis', vmin=0, vmax=v_max)
            axes[row, 1].set_title(f'Prediction (Hour {t_idx+1})'); fig.colorbar(im2, ax=axes[row, 1])
            diff = pred_s - targ_s
            diff_max = np.max(np.abs(diff)) if np.max(np.abs(diff)) > 0 else 0.1
            im3 = axes[row, 2].imshow(diff, cmap='RdBu_r', vmin=-diff_max, vmax=diff_max)
            axes[row, 2].set_title(f'Error (Hour {t_idx+1})'); fig.colorbar(im3, ax=axes[row, 2])
        plt.tight_layout(rect=[0, 0, 1, 0.97])
        plt.savefig(f"{save_path}_{i}.png", dpi=200)
        plt.show()
    print(f"✅ Comparison maps saved to '{save_path}_X.png'")

# --- Run Evaluation and Generate Visualizations ---
vis_data, all_preds_flat, all_targets_flat = evaluate_and_visualize(model, test_loader, device, scalers, TARGET_VAR)

print("\n--- Generating Visualization 1: Comparison Maps ---")
plot_visual_comparison(vis_data, save_path=f'test_comparison_lookback_{LOOKBACK_HOURS}', timesteps_to_plot=[0, 23, 71, 167])

print("\n--- Generating Visualization 2 & 3: Scatter and Error Plots ---")
errors = all_preds_flat - all_targets_flat

plt.figure(figsize=(8, 8))
sample_indices = np.random.choice(len(all_preds_flat), min(len(all_preds_flat), 10000), replace=False)
plt.scatter(all_targets_flat[sample_indices], all_preds_flat[sample_indices], alpha=0.3, s=10)
min_val, max_val = min(all_targets_flat.min(), all_preds_flat.min()), max(all_targets_flat.max(), all_preds_flat.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')
plt.xlabel("Actual Wave Height (m)")
plt.ylabel("Predicted Wave Height (m)")
plt.title(f"Scatter Plot (Lookback: {LOOKBACK_HOURS}hrs)")
plt.grid(True)
plt.legend()
plt.axis('equal')
plt.tight_layout()
plt.savefig(f'scatter_plot_lookback_{LOOKBACK_HOURS}.png', dpi=300)
plt.show()
print("✅ Scatter plot saved.")

mean_error = np.mean(errors)
plt.figure(figsize=(10, 6))
plt.hist(errors, bins=100, density=True)
plt.axvline(mean_error, color='r', linestyle='--', lw=2, label=f'Mean Error: {mean_error:.3f}')
plt.title(f"Distribution of Prediction Errors (Lookback: {LOOKBACK_HOURS}hrs)")
plt.xlabel("Error (Predicted - Actual) in meters")
plt.ylabel("Density")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(f'error_histogram_lookback_{LOOKBACK_HOURS}.png', dpi=300)
plt.show()
print("✅ Error histogram saved.")

print("\n✅ All evaluation visualizations are complete.")